In [3]:
import numpy as np
import re
import pickle
import gzip

import sys
sys.path.append('/Users/chenwei/Desktop/Github/ViDa') 

import imp, vida.data_processing.utils
imp.reload(vida.data_processing.utils)
from vida.data_processing.utils import *

import imp, vida.adjmat.dp2adj
imp.reload(vida.adjmat.dp2adj)
from vida.adjmat.dp2adj import *

import imp, vida.data_processing.comp_time
imp.reload(vida.data_processing.comp_time)
from vida.data_processing.comp_time import *


### three way

In [ ]:
fpath = '/Users/chenwei/Desktop/Github/ViDa//data/raw_data/Machinek-data'
rxn = 'Machinek-PRF'
ref_strands = 'CCCTCCACATTCAACCTCAAACTCACC+TGGTGTTTGTGGGTGTGGTGAGTTTGAGGTTGA+GGTGAGTTTGAGGTTGAATGTGGA'
strand_a = 'CCCTCCACATTCAACCTCAAACTCACC'  # substrate_perf_seq
strand_b = 'TGGTGTTTGTGGGTGTGGTGAGTTTGAGGTTGA'  # incumbent_perf_seq
strand_c = 'GGTGAGTTTGAGGTTGAATGTGGA'  # invader_perf_seq
strand_list = [strand_a, strand_b, strand_c]

ref_name_list = assign_base_names(ref_strands)
ref_name = [item for sublist in ref_name_list for item in sublist]


In [ ]:
strand_list

In [ ]:
# f = open('/Users/chenwei/Desktop/Github/ViDa/data/raw_data/Machinek-data/Machinek-PRF/Machinek-PRF-0.txt', 'r')
# lines = f.read().splitlines()
# part_strand = 'GGTGAGTTTGAGGTTGAATGTGGA'
# seq_line = []
# pattern = re.compile(r'^(.*?)\s+t=([\d.e+-]+) seconds, dG=([-\d.e+]+) kcal/mol')

### read files

In [ ]:
trajs_seqs, trajs_states, trajs_times, trajs_energies = read_machinek(fpath,rxn,strand_a,num_files=1)
trajs_seqs.shape, trajs_states.shape

In [ ]:
trajs_seqs[0][-10:]

### preprocess

In [ ]:
dp_arr, dp_og, pair, energy, trans_time = concat_machinek(trajs_states, trajs_times, trajs_energies)
dp_uniq, dp_og_uniq, pair_uniq, energy_uniq, indices_uniq, indices_all = get_uniq(dp_arr, dp_og, pair, energy)
adj_uniq, seqlabel_uniq = sim_adj_3strand_uniq(dp_arr, trajs_seqs, ref_name, ref_name_list, strand_list, indices_uniq)    

#### adj_uqni debug

In [ ]:
dp_arr.shape, dp_arr[0].shape, trajs_seqs.shape, len(trajs_seqs[0]), adj_uniq.shape, seqlabel_uniq.shape, indices_all.shape

In [ ]:
la = 980
lb = la+10

seqlabel_uniq[la:lb], dp_og_uniq[la:lb]

In [ ]:
dp_og_uniq[-1], adj_uniq[-1]

In [ ]:
dp_og_uniq[-1], dp_uniq[-1], seqlabel_uniq[-1]

In [ ]:
"""
## TEST 1
## Calculate the adjacency matrix for the individual structure ##

"""
dp_structure = dp_uniq[-1]

alter_name_arr = concat_disorder(trajs_seqs[0],ref_name_list, strand_list)

alter_name_arr_test = alter_name_arr[-1]

alter_name = np.concatenate(alter_name_arr_test)

dplast = dp2adj_3strand(ref_name, alter_name, alter_name_arr_test, dp_structure)
dplast

In [ ]:
"""
## TEST 2
## Calculate the whole adjacency matrix than compared with uniq ones ##

"""
dppmat = sim_adj_3strand(dp_arr, trajs_sequences, ref_name, ref_name_list, strand_list)
dppmat_uniq = dppmat[indices_uniq]

np.all(adj_uniq == dppmat_uniq)

### UNCHANGE

### collect time

In [ ]:
hold_time, trj_id = sim_ht(trans_time)
hold_time_uniq = mean_holdingtime(hold_time, indices_uniq, indices_all)
cum_time_uniq,freq_uniq = cumu_holdingtime(hold_time, indices_uniq, indices_all)


### dp to adj

In [ ]:
adj_uniq = sim_adj_3strand_uniq(dp_arr, trajs_seqs, ref_name, ref_name_list, strand_list, indices_uniq)    


In [ ]:
combined_array = np.concatenate(adj, axis=0)
combined_array.shape

### Draw

In [ ]:
import networkx as nx

import matplotlib.pyplot as plt

# Create a graph from the adjacency matrix
G = nx.Graph(adj[10])

# Draw the graph
pos = nx.spring_layout(G)  # Positions for all nodes
nx.draw(G, pos=nx.kamada_kawai_layout(G), with_labels=True, font_weight='bold', node_size=200, node_color='green', font_size=10, font_color='black')
# nx.draw(G, with_labels=True, font_weight='bold', node_size=200, node_color='green', font_size=10, font_color='black')


# Show the plot
plt.show()


### Debug

In [ ]:
import numpy as np
import re
import pickle
import gzip
import pandas as pd
import sys
sys.path.append('/Users/chenwei/Desktop/Github/ViDa') 


import imp, vida.data_processing.strandReorder
imp.reload(vida.data_processing.strandReorder)
from vida.data_processing.strandReorder import *

import imp, vida.data_processing.utils
imp.reload(vida.data_processing.utils)
from vida.data_processing.utils import *

import imp, vida.adjmat.dp2adj
imp.reload(vida.adjmat.dp2adj)
from vida.adjmat.dp2adj import *

import imp, vida.data_processing.comp_time
imp.reload(vida.data_processing.comp_time)
from vida.data_processing.comp_time import *


In [4]:
datafile='Machinek-PRF-trunc'

ref_strands = 'CCCTCCACATTCAACCTCAAACTCACC+TGGTGTTTGTGGGTGTGGTGAGTTTGAGGTTGA+GGTGAGTTTGAGGTTGAATGTGGA'
strand_sub = 'CCCTCCACATTCAACCTCAAACTCACC'  # substrate_perf_seq
strand_incb = 'TGGTGTTTGTGGGTGTGGTGAGTTTGAGGTTGA'  # incumbent_perf_seq
strand_inv = 'GGTGAGTTTGAGGTTGAATGTGGA'  # invader_perf_seq

In [5]:
inpath = f'../data/post_data/{datafile}/preprocess_Machinek-PRF.npz'
loaded_data = np.load(inpath, allow_pickle=True)
dp_uniq = loaded_data['dp_uniq']
dp_og_uniq = loaded_data['dp_og_uniq']
shortname_uniq = loaded_data['shortname_uniq']
incbinvpair_uniq = loaded_data['incbinvpair_uniq']
indices_uniq = loaded_data['indices_uniq']
indices_all = loaded_data['indices_all']
trans_time = loaded_data["trans_time"]
energy_uniq = loaded_data["energy_uniq"]
ref_name = loaded_data["ref_name"]
ref_name_list = loaded_data["ref_name_list"]
strand_list = loaded_data["strand_list"]

inpath2 = f'../data/post_data/{datafile}/Machinek-PRF.pkl.gz'
with gzip.open(inpath2, 'rb') as file:
    load_data_seq = pickle.load(file)

ref_name = load_data_seq["ref_name"]
ref_name_list = load_data_seq["ref_name_list"]
strand_list = load_data_seq["strand_list"]
trajs_seqs = load_data_seq["trajs_seqs"]
trajs_states = load_data_seq['trajs_states']
trajs_times = load_data_seq['trajs_times']
trajs_energies = load_data_seq['trajs_energies']
trajs_shortnames = load_data_seq['trajs_shortnames']
trajs_incbinvpairs = load_data_seq['trajs_incbinvpairs']


In [5]:
datafile = 'Machinek-Mismatch14-trunc'
inpath2 = f'../data/post_data/{datafile}/Machinek-Mismatch14.pkl.gz'
with gzip.open(inpath2, 'rb') as file:
    load_data_seq = pickle.load(file)

ref_name = load_data_seq["ref_name"]
ref_name_list = load_data_seq["ref_name_list"]
strand_list = load_data_seq["strand_list"]
trajs_seqs = load_data_seq["trajs_seqs"]
trajs_states = load_data_seq['trajs_states']
trajs_times = load_data_seq['trajs_times']
trajs_energies = load_data_seq['trajs_energies']
trajs_shortnames = load_data_seq['trajs_shortnames']
trajs_incbinvpairs = load_data_seq['trajs_incbinvpairs']

In [13]:
df = pd.DataFrame(data={
                "Energy": energy_uniq, "DP": dp_og_uniq, 
                "IncbInvPair": incbinvpair_uniq, "ShortName": shortname_uniq,
                }
                )

In [21]:
color_mapping = {
        "inv+sub+incb": "grey",
        "incb+sub+inv": "red",
        "incb+sub inv": "blue",    
        "incb sub+inv": "orange",
    }
    

In [23]:
color_mapping[df['ShortName'][0]]

'grey'

In [7]:
unique_values, counts = np.unique(shortname_uniq, return_counts=True)
print(unique_values)  # Output: array of unique values
print(counts)  # Output: corresponding counts

['incb sub+inv' 'incb+sub inv' 'incb+sub+inv' 'inv+sub+incb']
[   32   611   178 15522]


In [20]:
len(trajs_seqs), len(trajs_shortnames), len(trajs_incbinvpairs), len(trajs_states), len(trajs_times), len(trajs_energies)

(400, 400, 400, 400, 400, 400)

In [44]:
dp_uniq.shape, dp_og_uniq.shape, shortname_uniq.shape, incbinvpair_uniq.shape, indices_uniq.shape, indices_all.shape, trans_time.shape

((16343,), (16343,), (16343,), (16343,), (16343,), (4363808,), (4363808,))

In [23]:
dp, dp_og, pair, energy, trans_time, seq, shortname, incbinvpair= concat_machinek(trajs_states, trajs_times, trajs_energies, trajs_seqs, trajs_shortnames, trajs_incbinvpairs)

dp.shape, pair.shape, dp_og.shape, energy.shape, trans_time.shape, seq.shape, shortname.shape, incbinvpair.shape,

((4363808,),
 (4363808,),
 (4363808,),
 (4363808,),
 (4363808,),
 (4363808,),
 (4363808,),
 (4363808,))

### dp2adj

In [89]:
inpath3 = f'../data/post_data/{datafile}/adjmat_Machinek-PRF.npz'
loaded_data3 = np.load(inpath3, allow_pickle=True)
adj_uniq = loaded_data3['adj_uniq']

adj_uniq.shape

(16343, 84, 84)

In [99]:
np.unique(adj_uniq.reshape(adj_uniq.shape[0], -1), axis=0).shape

(16343, 7056)

### Class

In [14]:
import re
from itertools import permutations
import numpy as np


class ThreeStrandReorder:
    
    @staticmethod
    def get_strand_pos(bpair):
        pattern = r'([a-z])(\d+)([a-z])(\d+)'
        match = re.match(pattern, bpair)

        if match:
            strand1, pos1, strand2, pos2 = match.groups()
            pos1 = int(pos1) - 1  # Convert to 0-based index
            pos2 = int(pos2) - 1  # Convert to 0-based index
            return strand1, pos1, strand2, pos2
        else:
            raise ValueError("Invalid base pair format")

    @staticmethod
    def hasIncbInvPair(base_pairs):
        has_bc = any('b' in pair and 'c' in pair for pair in base_pairs)

        return 1 if has_bc else 0
    
    
    def get_basepairs(self, seq, dp, ref_name_list, strand_list):
        def concat_disorder(seq, ref_name_list, strand_list):
            sequence_list = re.split(r'\s|\+', seq) 
            sequence = ''.join(sequence_list)

            for permuted_strand in permutations(strand_list):
                combined_sequence = ''.join(permuted_strand)
                if combined_sequence == sequence:
                    alter_name = [ref_name_list[strand_list.index(strand)] for strand in permuted_strand]
                    break

            return np.concatenate(alter_name)

        dp_structure = dp.replace(' ', '').replace('+', '')
        alter_name = concat_disorder(seq, ref_name_list, strand_list)

        stack = []
        base_pairs = []

        for name, char in zip(alter_name, dp_structure):
            if char == '(':
                stack.append(name)
            elif char == ')':
                if stack:
                    opening_index = stack.pop()
                    base_pairs.append(opening_index + name)
                else:
                    raise ValueError("Mismatched brackets")

        if stack:
            raise ValueError("Mismatched brackets")

        return base_pairs

    def reorderCase1(self, case1, dp, seq, short_seqname, ref_name_list, strand_list):
        """
        Reorder:
        'incb+inv+sub' -> 'inv+sub+incb' 
        'sub+incb+inv' -> 'inv+sub+incb'
        """
        
        position = case1.index(short_seqname)
        base_pairs = self.get_basepairs(seq, dp, ref_name_list, strand_list)
        
        incb_inv_pair = ThreeStrandReorder.hasIncbInvPair(base_pairs)
                
        if position == 0:   # 'inv+sub+incb': c+a+b ===> do nothing
            return dp, short_seqname, incb_inv_pair
        
        
        if position == 1:  # 'incb+inv+sub': b+c+a ===> c+a+b
            incb_list, inv_list, sub_list = [list(part) for part in re.split(r'\s|\+', dp)]
            
            for bpair in base_pairs:
                strand1, pos1, strand2, pos2 = ThreeStrandReorder.get_strand_pos(bpair)
                
                # only need to change incb (b) with its connected strands
                if strand1 == 'b' and strand2 == 'c':
                    incb_list[pos1] = ')'
                    inv_list[pos2] = '('

                if strand1 == 'b' and strand2 == 'a':
                    incb_list[pos1] = ')'
                    sub_list[pos2] = '('


        if position == 2: # 'sub+incb+inv': a+b+c ===> c+a+b
            sub_list, incb_list, inv_list = [list(part) for part in re.split(r'\s|\+', dp)]
            
            for bpair in base_pairs:
                strand1, pos1, strand2, pos2 = ThreeStrandReorder.get_strand_pos(bpair)
            
                # only need to change inv (c) with its connected strands
                if strand1 == 'a' and strand2 == 'c':
                    inv_list[pos2] = '('
                    sub_list[pos1] = ')'
                                    
                if strand1 == 'b' and strand2 == 'c':
                    inv_list[pos2] = '('
                    incb_list[pos1] = ')'
        
        inv = ''.join(inv_list)
        sub = ''.join(sub_list)
        incb = ''.join(incb_list)
        # inv+sub+incb
        dp_new = inv + "+" + sub + "+" + incb
        
        return dp_new, case1[0], incb_inv_pair


    def reorderCase2(self, case2, dp, seq, short_seqname, ref_name_list, strand_list):
        """
        Reorder:
        'inv+incb+sub' -> 'incb+sub+inv' 
        'sub+inv+incb' -> 'incb+sub+inv'
        """
        
        position = case2.index(short_seqname)
        base_pairs = self.get_basepairs(seq, dp, ref_name_list, strand_list)
        
        incb_inv_pair = ThreeStrandReorder.hasIncbInvPair(base_pairs)
                
        if position == 0:   # 'incb+sub+inv': b+a+c ===> do nothing
            return dp, short_seqname, incb_inv_pair
        
        if position == 1:  # 'inv+incb+sub': c+b+a ===> b+a+c
            inv_list, incb_list, sub_list = [list(part) for part in re.split(r'\s|\+', dp)]
            
            for bpair in base_pairs:
                strand1, pos1, strand2, pos2 = ThreeStrandReorder.get_strand_pos(bpair)
                
                # only need to change inv (c) with its connected strands
                if strand1 == 'c' and strand2 == 'b':
                    inv_list[pos1] = ')'
                    incb_list[pos2] = '('
                    incb_inv_pair = 1
                    
                if strand1 == 'c' and strand2 == 'a':
                    inv_list[pos1] = ')'
                    sub_list[pos2] = '('
        
        
        if position == 2: # 'sub+inv+incb': a+c+b ===> b+a+c
            sub_list, incb_list, inv_list = [list(part) for part in re.split(r'\s|\+', dp)]
            
            for bpair in base_pairs:
                strand1, pos1, strand2, pos2 = ThreeStrandReorder.get_strand_pos(bpair)
            
                # only need to change incb (b) with its connected strands
                if strand1 == 'a' and strand2 == 'b':
                    incb_list[pos2] = '('
                    sub_list[pos1] = ')'
                    
                if strand1 == 'c' and strand2 == 'b':
                    incb_list[pos2] = '('
                    inv_list[pos1] = ')'
        
        inv = ''.join(inv_list)
        sub = ''.join(sub_list)
        incb = ''.join(incb_list)
        # incb+sub+inv
        dp_new = incb + "+" + sub + "+" + inv
        
        return dp_new, case2[0], incb_inv_pair


    def reorderCase3(self, case3, dp, seq, short_seqname, ref_name_list, strand_list):
        """
        Reorder:
        'sub+incb inv' -> 'incb+sub inv' 
        'inv sub+incb' -> 'incb+sub inv'
        'sub+inv incb' -> 'incb+sub inv' 
        """
        
        position = case3.index(short_seqname)
        
        incb_inv_pair = 0 # certainly no incumbent-invader pair
                
        if position == 0:   # 'incb+sub inv': b+a c ===> do nothing
            return dp, short_seqname, incb_inv_pair
        
        
        base_pairs = self.get_basepairs(seq, dp, ref_name_list, strand_list)
        
        if position == 1:  # 'sub+incb inv': a+b c ===> b+a c
            sub_list, incb_list, inv_list  = [list(part) for part in re.split(r'\s|\+', dp)]
            
            for bpair in base_pairs:
                strand1, pos1, strand2, pos2 = ThreeStrandReorder.get_strand_pos(bpair)
                
                # only need to change incb (b) with its connected strand sub (a)
                if strand1 == 'a' and strand2 == 'b':
                    incb_list[pos2] = '('
                    sub_list[pos1] = ')'
        
        
        if position == 2: # 'inv sub+incb': c a+b ===> b+a c
            inv_list, sub_list, incb_list = [list(part) for part in re.split(r'\s|\+', dp)]
            
            for bpair in base_pairs:
                strand1, pos1, strand2, pos2 = ThreeStrandReorder.get_strand_pos(bpair)
                
                # only need to change incb (b) with its connected strand sub (a)
                if strand1 == 'a' and strand2 == 'b':
                    incb_list[pos2] = '('
                    sub_list[pos1] = ')'
                    
        
        if position == 3: # 'inv incb+sub': c b+a ===> b+a c
            inv_list, incb_list, sub_list = [list(part) for part in re.split(r'\s|\+', dp)]
            # just need to exchange the position of inv and incb+sub
            
        inv = ''.join(inv_list)
        sub = ''.join(sub_list)
        incb = ''.join(incb_list)
        # incb+sub inv
        dp_new = incb + "+" + sub + " " + inv
        
        return dp_new, case3[0], incb_inv_pair


    def reorderCase4(self, case4, dp, seq, short_seqname, ref_name_list, strand_list):
        """
        Reorder:
        'incb inv+sub' -> 'incb sub+inv' 
        'inv+sub incb' -> 'incb sub+inv' 
        'sub+inv incb' -> 'incb sub+inv' 
        """
        
        position = case4.index(short_seqname)
        
        incb_inv_pair = 0 # certainly no incumbent-invader pair
                
        if position == 0:   # 'incb sub+inv': b a+c ===> do nothing
            return dp, short_seqname, incb_inv_pair
        
        
        base_pairs = self.get_basepairs(seq, dp, ref_name_list, strand_list)
        
        if position == 1:  # 'incb inv+sub': b c+a ===> b a+c
            incb_list, inv_list, sub_list = [list(part) for part in re.split(r'\s|\+', dp)]
            
            for bpair in base_pairs:
                strand1, pos1, strand2, pos2 = ThreeStrandReorder.get_strand_pos(bpair)
                
                # only need to change inv (c) with its connected strand sub (a)
                if strand1 == 'c' and strand2 == 'a':
                    inv_list[pos1] = ')'
                    sub_list[pos2] = '('
                    
        
        if position == 2: # 'inv+sub incb': c+a b ===> b a+c
            inv_list, sub_list, incb_list = [list(part) for part in re.split(r'\s|\+', dp)]
            
            for bpair in base_pairs:
                strand1, pos1, strand2, pos2 = ThreeStrandReorder.get_strand_pos(bpair)
                
                # only need to change inv (c) with its connected strand sub (a)
                if strand1 == 'c' and strand2 == 'a':
                    inv_list[pos1] = ')'
                    sub_list[pos2] = '('
        
        
        if position == 3: # 'sub+inv incb': a+c b ===> b a+c
            sub_list, inv_list, incb_list = [list(part) for part in re.split(r'\s|\+', dp)]
            # just need to exchange the position of sub+inv and incb
            
        inv = ''.join(inv_list)
        sub = ''.join(sub_list)
        incb = ''.join(incb_list)
        # incb sub+inv
        dp_new = incb + " " + sub + "+" + inv
        
        return dp_new, case4[0], incb_inv_pair


    def reorderCase5(self, case5, dp, seq, short_seqname, ref_name_list, strand_list):
        """
        Reorder:
        'inv+incb sub' -> 'incb+inv sub' 
        'sub inv+incb' -> 'incb+inv sub' 
        'sub incb+inv' -> 'incb+inv sub' 
        """
        
        position = case5.index(short_seqname)
        
        incb_inv_pair = 1 # certainly having incumbent-invader pair
                
        if position == 0:   # 'incb+inv sub': b+c a ===> do nothing
            return dp, short_seqname, incb_inv_pair
        
        
        base_pairs = self.get_basepairs(seq, dp, ref_name_list, strand_list)
        
        if position == 1:  # 'inv+incb sub': c+b a ===> b+c a
            inv_list, incb_list, ub_list = [list(part) for part in re.split(r'\s|\+', dp)]
            
            for bpair in base_pairs:
                strand1, pos1, strand2, pos2 = ThreeStrandReorder.get_strand_pos(bpair)
                
                # only need to change inv (c) with its connected strand incb (b)
                if strand1 == 'c' and strand2 == 'b':
                    inv_list[pos1] = ')'
                    incb_list[pos2] = '('
                    
        
        if position == 2: # 'sub inv+incb': a c+b ===> b+c a
            sub_list, inv_list, incb_list = [list(part) for part in re.split(r'\s|\+', dp)]
            
            for bpair in base_pairs:
                strand1, pos1, strand2, pos2 = ThreeStrandReorder.get_strand_pos(bpair)
                
                # only need to change inv (c) with its connected strand incb (b)
                if strand1 == 'c' and strand2 == 'b':
                    inv_list[pos1] = ')'
                    incb_list[pos2] = '('
        
        
        if position == 3: # 'sub incb+inv': a b+c ===> b+c a
            sub_list, incb_list, inv_list = [list(part) for part in re.split(r'\s|\+', dp)]
            # just need to exchange the position of sub and incb+inv
            
        inv = ''.join(inv_list)
        sub = ''.join(sub_list)
        incb = ''.join(incb_list)
        # incb+inv sub
        dp_new = incb + "+" + inv + " " + sub
        
        return dp_new, case5[0], incb_inv_pair


    def reorderCase6(self, case6, dp, seq, short_seqname, ref_name_list, strand_list):
        """
        Reorder:
        'incb inv sub' -> 'incb sub inv' 
        'sub inv incb' -> 'incb sub inv' 
        'sub incb inv' -> 'incb sub inv' 
        'inv incb sub' -> 'incb sub inv' 
        'inv sub incb' -> 'incb sub inv' 
        """
        
        position = case6.index(short_seqname)
        
        incb_inv_pair = 0 # certainly not incumbent-invader pair
                
        if position == 0:   # 'incb sub inv': b a c ===> do nothing
            return dp, short_seqname, incb_inv_pair
            
        if position == 1:  # 'incb inv sub': b c a ===> b a c
            incb_list, inv_list, sub_list = [list(part) for part in re.split(r'\s|\+', dp)]
        
        if position == 2: # 'sub inv incb': a c b ===> b a c
            sub_list, inv_list, incb_list = [list(part) for part in re.split(r'\s|\+', dp)]
        
        if position == 3: # 'sub incb inv': a b c ===> b a c
            sub_list, incb_list, inv_list = [list(part) for part in re.split(r'\s|\+', dp)]

        if position == 4: # 'inv incb sub': c b a ===> b a c
            inv_list, incb_list, sub_list = [list(part) for part in re.split(r'\s|\+', dp)]
        
        if position == 5: # 'inv sub incb': c a b ===> b a c
            inv_list, sub_list, incb_list = [list(part) for part in re.split(r'\s|\+', dp)]
                
        inv = ''.join(inv_list)
        sub = ''.join(sub_list)
        incb = ''.join(incb_list)
        # incb sub inv
        dp_new = incb + " " + sub + " " + inv
        
        return dp_new, case6[0], incb_inv_pair
    
    def nameMap(self, sequence, strand_sub, strand_incb, strand_inv):
        def replace_strings(s):
            s = s.replace(strand_inv, "inv")
            s = s.replace(strand_incb, "incb")
            s = s.replace(strand_sub, "sub")
            return s

        vectorized_replace = np.vectorize(replace_strings)

        return vectorized_replace(np.array([sequence])).tolist()[0]
    
    def dp_reorder(self, dp, seq, ref_name_list, strand_list, strand_sub, strand_incb, strand_inv):
    
        short_seqname = self.nameMap(seq, strand_sub, strand_incb, strand_inv)
        
        case1 = ['inv+sub+incb', 'incb+inv+sub', 'sub+incb+inv']
        case2 = ['incb+sub+inv', 'inv+incb+sub', 'sub+inv+incb']
        case3 = ['incb+sub inv', 'sub+incb inv', 'inv sub+incb', 'inv incb+sub'] 
        case4 = ['incb sub+inv', 'incb inv+sub', 'inv+sub incb', 'sub+inv incb'] 
        case5 = ['incb+inv sub', 'inv+incb sub', 'sub inv+incb', 'sub incb+inv'] ##
        case6 = ['incb sub inv', 'incb inv sub', 'sub inv incb' 'sub incb inv' 'inv incb sub' 'inv sub incb']
        
        if short_seqname in case1:
            dp_new, short_seqname, incb_inv_pair = self.reorderCase1(case1, dp, seq, short_seqname, ref_name_list, strand_list)
            
        if short_seqname in case2:
            dp_new, short_seqname, incb_inv_pair = self.reorderCase2(case2, dp, seq, short_seqname, ref_name_list, strand_list)
        
        if short_seqname in case3:
            dp_new, short_seqname, incb_inv_pair = self.reorderCase3(case3, dp, seq, short_seqname, ref_name_list, strand_list)
            
        if short_seqname in case4:
            dp_new, short_seqname, incb_inv_pair = self.reorderCase4(case4, dp, seq, short_seqname, ref_name_list, strand_list)
        
        if short_seqname in case5:
            dp_new, short_seqname, incb_inv_pair = self.reorderCase5(case5, dp, seq, short_seqname, ref_name_list, strand_list)
        
        if short_seqname in case6:
            dp_new, short_seqname, incb_inv_pair = self.reorderCase6(case6, dp, seq, short_seqname, ref_name_list, strand_list)
        
        return dp_new, short_seqname, incb_inv_pair



In [16]:
reoder_3strand = ThreeStrandReorder()

num = 1560

seq = seqlabel_uniq[num]
dp = dp_og_uniq[num]
print(arr_short[num], dp)
print()

p_new, short_seqname, incb_inv_pair = reoder_3strand.dp_reorder(dp, seq, ref_name_list, strand_list, strand_sub, strand_incb, strand_inv)
print(p_new, short_seqname, incb_inv_pair)


sub+incb+inv ((....((..(((((((((((((((((+................)))))))))))))))))+......((((....)))).)))).

......((((....)))).((((.+))....))..(((((((((((((((((+................))))))))))))))))) inv+sub+incb 0


In [ ]:
case1 = ['inv+sub+incb', 'sub+incb+inv', 'incb+inv+sub',]
case2 = ['incb+sub+inv', 'inv+incb+sub', 'sub+inv+incb']
case3 = ['incb+sub inv', 'sub+incb inv', 'inv sub+incb', 'inv incb+sub'] 
case4 = ['incb sub+inv', 'incb inv+sub', 'inv+sub incb', 'sub+inv incb'] 
case5 = ['incb+inv sub', 'inv+incb sub', 'sub inv+incb', 'sub incb+inv'] ##

np.where(arr_short == case1[0])[0]

## Load data for debugging

In [128]:
import numpy as np
import pickle
import gzip

nnn = 'new300'

fdata0 = f'../data/post_data/perfect_toehold8/{nnn}/perfect_toehold8.pkl.gz'
with gzip.open(fdata0, 'rb') as file:
    load_data = pickle.load(file)
fdata1 = f'../data/post_data/perfect_toehold8/{nnn}/preprocess_perfect_toehold8.npz'
fdata2 = f'../data/post_data/perfect_toehold8/{nnn}/time_perfect_toehold8.npz'

data1 = np.load(fdata1, allow_pickle=True)
data2 = np.load(fdata2, allow_pickle=True)
data1.files, data2.files, load_data.keys()

(['dp_uniq',
  'dp_og_uniq',
  'energy_uniq',
  'indices_uniq',
  'indices_all',
  'trans_time',
  'ref_name'],
 ['hold_time', 'hold_time_uniq', 'cum_time_uniq', 'freq_uniq', 'trj_id'],
 dict_keys(['trajs_states', 'trajs_times', 'trajs_energies', 'ref_name']))

In [135]:
data1['dp_uniq'].shape, len(data1['indices_all']), data1['trans_time'].shape, data2['hold_time'].shape

((128252,), 18249877, (18250177,), (18250177,))

In [134]:
indices_all = data1['indices_all']
trj_id = data2['trj_id']
len(trj_id), len(indices_all)

(300, 18249877)

In [107]:
len(trj_id), trj_id[-1]

(300, 18250176)

In [124]:
data2['hold_time'][18250176], data1['trans_time'][18250176], data1['trans_time'][-1]

(0.0, 3.197048082813411e-11, 3.197048082813411e-11)

In [74]:
import numpy as np
import pickle
import gzip



fdata0 = f'../data/post_data/perfect_toehold8/perfect_toehold8.pkl.gz'
with gzip.open(fdata0, 'rb') as file:
    load_data = pickle.load(file)
fdata1 = f'../data/post_data/perfect_toehold8/preprocess_perfect_toehold8.npz'
fdata2 = f'../data/post_data/perfect_toehold8/time_perfect_toehold8.npz'
print(load_data.keys())
data1 = np.load(fdata1, allow_pickle=True)
data2 = np.load(fdata2, allow_pickle=True)
print(data1.files, data2.files, load_data.keys())
trj_id = data2['trj_id']
indices_all = data1['indices_all']
hold_time_uniq = data2['hold_time_uniq']
trajs_states = load_data['trajs_states']
dp_og_uniq = data1['dp_og_uniq']
energy_uniq = data1['energy_uniq']
hold_time = data2['hold_time']
trans_time = data1['trans_time']
data1['dp_uniq'].shape, len(data1['indices_all']), data1['trans_time'].shape, data2['hold_time'].shape

dict_keys(['trajs_states', 'trajs_times', 'trajs_energies', 'ref_name'])
['dp_uniq', 'dp_og_uniq', 'energy_uniq', 'indices_uniq', 'indices_all', 'trans_time', 'ref_name'] ['hold_time', 'hold_time_uniq', 'cum_time_uniq', 'freq_uniq', 'trj_id'] dict_keys(['trajs_states', 'trajs_times', 'trajs_energies', 'ref_name'])


((128252,), 18249877, (18249877,), (18249877,))

In [75]:
data1['ref_name']

array(['a1', 'a2', 'a3', 'a4', 'a5', 'a6', 'a7', 'a8', 'a9', 'a10', 'a11',
       'a12', 'a13', 'a14', 'a15', 'a16', 'b1', 'b2', 'b3', 'b4', 'b5',
       'b6', 'b7', 'b8', 'b9', 'b10', 'b11', 'b12', 'b13', 'b14', 'b15',
       'b16', 'b17', 'b18', 'b19', 'b20', 'b21', 'b22', 'b23', 'b24',
       'c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9', 'c10', 'c11',
       'c12', 'c13', 'c14', 'c15', 'c16', 'c17', 'c18', 'c19', 'c20',
       'c21', 'c22', 'c23', 'c24', 'c25', 'c26'], dtype='<U3')

In [74]:
dp_og = dp_og_uniq[indices_all]
energy = energy_uniq[indices_all]

arrays_to_split = [dp_og, trans_time, energy]
subtrj_id = (trj_id+1)[:-1]
sub_arrays = [np.split(arr, subtrj_id) for arr in arrays_to_split]

In [96]:
sub_dp_og, sub_trans_time, sub_energy = [], [], []

for i in range(len(sub_arrays[0])):
    dp_og_i = sub_arrays[0][i]
    trans_time_i = sub_arrays[1][i]
    energy_i = sub_arrays[2][i]
    dp_i_unique, idx = np.unique(dp_og_i, axis=0, return_index=True)
    energy_i_unique = energy_i[idx]
    sub_dp_og.append(dp_i_unique)
    sub_trans_time.append(trans_time_i)
    sub_energy.append(energy_i_unique)
    

In [98]:
sub_trans_time[0].shape, sub_energy[0].shape, sub_dp_og[0].shape

((161430,), (6069,), (6069,))

In [100]:
sub_trans_time[0].shape, sub_energy[0].shape, sub_dp_og[0].shape

((21,), (6069,), (6069,))

In [95]:
sub_arrays[0][0].shape

(161430,)

In [2]:
length = 0
for ll in load_data['trajs_times']:
    length += len(ll)
    # print( len(ll))
length

18249877

In [3]:
length = 0
for trj in trajs_states:
    length+=len(trj)
length

18249877

In [4]:
import networkx as nx

# Build the edges
def get_all_edges(indices_all, trj_id):
    all_nodes = indices_all
    all_edges = []
    
    # Create a boolean mask to track which edges to keep
    keep_edge = [True] * (len(all_nodes) - 1)
    
    # Mark edges to delete
    for idx in trj_id[:-1]:
        keep_edge[idx] = False
    
    # Remove the deleted edges
    for i, (previous, current) in enumerate(zip(all_nodes, all_nodes[1:])):
        if keep_edge[i]:
            all_edges.append((previous, current))
    
    return all_edges


# construct weighted directed graph
def build_wdg(all_edges, hold_time_uniq):
    DG = nx.DiGraph()
    for i in range(len(all_edges)):
        weight = hold_time_uniq[all_edges[i][0]]
        DG.add_edge(int(all_edges[i][0]), int(all_edges[i][1]), weight=float(weight))
    
    return DG  


def build_wdg2222(all_edges, hold_time_uniq):
    DG = nx.DiGraph()
    
    # Explicitly add all nodes here
    all_nodes = set()
    for edge in all_edges:
        all_nodes.add(int(edge[0]))
        all_nodes.add(int(edge[1]))
    
    # Add missing nodes (if you know they should exist)
    max_node = max(all_nodes)
    for node_id in range(max_node + 1):
        if node_id not in all_nodes:
            DG.add_node(node_id)
    
    # Add edges as before
    for i in range(len(all_edges)):
        weight = hold_time_uniq[all_edges[i][0]]
        DG.add_edge(int(all_edges[i][0]), int(all_edges[i][1]), weight=float(weight))
    
    return DG


In [10]:
all_edges000 = get_all_edges(indices_all, trj_id)
len(all_edges000), len(np.unique(all_edges000)), len(np.unique(indices_all))

(18249577, 128233, 128252)

In [11]:
DG = build_wdg(all_edges000, hold_time_uniq)
len(DG), max(DG.nodes), len(DG.edges)

(128233, 128251, 273080)

In [12]:
DG000 = build_wdg2222(all_edges000, hold_time_uniq)
len(DG000), max(DG000.nodes), len(DG000.edges)

(128252, 128251, 273080)

In [13]:
max_node = max(DG.nodes)
DG.nodes, DG[max_node], DG.has_edge(max_node,max_node), list(DG.in_edges(max_node))

(NodeView((26999, 26338, 5394, 5427, 5422, 3789, 26909, 26962, 26986, 26567, 26899, 26973, 26079, 26929, 26893, 26984, 24509, 24381, 5035, 22834, 4598, 4023, 24387, 24483, 24366, 24369, 26511, 26587, 24923, 23692, 26442, 5542, 25462, 25604, 25398, 23951, 19877, 36771, 28712, 36822, 20039, 20038, 20049, 25596, 24020, 24641, 5095, 24137, 24081, 26092, 25920, 25244, 25348, 23940, 4825, 23809, 25192, 25240, 23855, 23852, 25204, 25140, 4799, 23786, 4796, 23791, 25209, 5310, 9951, 25216, 15070, 15080, 15075, 15336, 15328, 25305, 23912, 25322, 25323, 12164, 12155, 10054, 25200, 25231, 25320, 12156, 98889, 99479, 99532, 99445, 98923, 99480, 98867, 90864, 98890, 98882, 98313, 98321, 90461, 96016, 104079, 98340, 98319, 90460, 90472, 98301, 90762, 90754, 90755, 22644, 22606, 3903, 4544, 22575, 3927, 22605, 4543, 3902, 17370, 8976, 773, 9638, 964, 9627, 4560, 22601, 14915, 34158, 28274, 25308, 9262, 23870, 23780, 25154, 25259, 23824, 23767, 23856, 25247, 20806, 14085, 15931, 15913, 2092, 15915, 14

In [14]:
max_node = max(DG000.nodes)
DG000.nodes, DG000[max_node], DG000.has_edge(max_node,max_node), list(DG000.in_edges(max_node))

(NodeView((6918, 12824, 13207, 13237, 13240, 13414, 13508, 13541, 13565, 13575, 17481, 18332, 19098, 19421, 23662, 23715, 23734, 25795, 26654, 26999, 26338, 5394, 5427, 5422, 3789, 26909, 26962, 26986, 26567, 26899, 26973, 26079, 26929, 26893, 26984, 24509, 24381, 5035, 22834, 4598, 4023, 24387, 24483, 24366, 24369, 26511, 26587, 24923, 23692, 26442, 5542, 25462, 25604, 25398, 23951, 19877, 36771, 28712, 36822, 20039, 20038, 20049, 25596, 24020, 24641, 5095, 24137, 24081, 26092, 25920, 25244, 25348, 23940, 4825, 23809, 25192, 25240, 23855, 23852, 25204, 25140, 4799, 23786, 4796, 23791, 25209, 5310, 9951, 25216, 15070, 15080, 15075, 15336, 15328, 25305, 23912, 25322, 25323, 12164, 12155, 10054, 25200, 25231, 25320, 12156, 98889, 99479, 99532, 99445, 98923, 99480, 98867, 90864, 98890, 98882, 98313, 98321, 90461, 96016, 104079, 98340, 98319, 90460, 90472, 98301, 90762, 90754, 90755, 22644, 22606, 3903, 4544, 22575, 3927, 22605, 4543, 3902, 17370, 8976, 773, 9638, 964, 9627, 4560, 22601, 1

In [15]:
list(nx.isolates(DG)), list(nx.isolates(DG000)), list(DG000.in_edges(6918))

([],
 [6918,
  12824,
  13207,
  13237,
  13240,
  13414,
  13508,
  13541,
  13565,
  13575,
  17481,
  18332,
  19098,
  19421,
  23662,
  23715,
  23734,
  25795,
  26654],
 [])

In [ ]:
all_nodes = indices_all
all_edges_temp = []

for previous, current in zip(all_nodes, all_nodes[1:]):
    all_edges_temp.append((previous, current))

indices_to_delete = trj_id[:-1]

In [20]:
indices_to_delete

array([  161429,   161450,   161477,   161478,   161798,   221192,
         629286,   629287,   630126,   631165,   631166,   631175,
         631206,   631274,   631750,   631771,   690561,   694744,
         696952,   696957,   696970,   698758,   698764,   700464,
         700487,  1117849,  1117850,  1120763,  1120828,  1120829,
        1120830,  1120871,  1120872,  1129341,  1129412,  1488498,
        1489545,  1517032,  1517219,  1517220,  1517221,  1517222,
        1517245,  1782953,  1787232,  1787237,  1793907,  1793908,
        2556701,  2556782,  2556873,  2926959,  2927655,  3136251,
        3138284,  3408425,  3442978,  3442979,  3443304,  3443351,
        3443550,  3446889,  3520390,  3520411,  3520644,  3520651,
        3520654,  3520677,  3520703,  3524313,  3524320,  3524366,
        3524814,  3524817,  3524951,  3524972,  3525008,  3525009,
        3525010,  3527288,  3527414,  3527415,  3653748,  3701760,
        4069419,  4069420,  4069490,  4069779,  4069784,  4069

In [216]:
DG = build_wdg(all_edges, hold_time_uniq)

In [217]:
len(DG), DG.nodes

(13316,
 NodeView((2377, 2260, 434, 437, 436, 290, 2357, 2367, 2373, 2276, 2355, 2368, 2229, 2362, 2354, 2372, 2047, 2028, 409, 1908, 371, 312, 2029, 2046, 2025, 2026, 2270, 2280, 2075, 1964, 2266, 447, 2140, 2151, 2138, 1982, 1625, 3400, 2550, 3403, 1632, 1631, 1633, 2148, 1986, 2057, 415, 2001, 1996, 2232, 2219, 2120, 2133, 1981, 382, 1972, 2110, 2119, 1976, 1975, 2113, 2103, 381, 1970, 380, 1971, 2114, 428, 806, 2115, 1272, 1274, 1273, 1294, 1293, 2125, 1980, 2128, 2129, 986, 984, 822, 2112, 2117, 2127, 985, 10262, 10310, 10316, 10308, 10267, 10311, 10259, 9683, 10263, 10260, 10206, 10208, 9645, 10053, 10699, 10211, 10207, 9644, 9646, 10204, 9678, 9676, 9677, 1882, 1879, 297, 363, 1875, 299, 1878, 362, 296, 1477, 717, 50, 759, 67, 758, 364, 1877, 1265, 3111, 2511, 2126, 739, 1978, 1969, 2104, 2122, 1973, 1968, 1977, 2121, 1702, 1181, 1337, 1335, 147, 1336, 1267, 1334, 1979, 1974, 379, 2116, 2118, 2130, 2124, 2106, 2109, 378, 9685, 9701, 9682, 10266, 10264, 9886, 9881, 10102, 10089, 

In [218]:
all_edges[0]

(2377, 2260)

In [201]:
DDDD = nx.DiGraph()
DDDD.nodes, DDDD.edges 

(NodeView(()), OutEdgeView([]))

In [202]:
DDDD.add_edge(int(all_edges[0][0]), int(all_edges[0][1]))

In [203]:
DDDD.nodes, DDDD.edges 

(NodeView((2381, 2264)), OutEdgeView([(2381, 2264)]))

13323

In [ ]:
import numpy as np
import pickle
import gzip

nnn = '10'

fdata0 = f'../data/post_data/perfect_toehold8/{nnn}/perfect_toehold8.pkl.gz'
with gzip.open(fdata0, 'rb') as file:
    load_data = pickle.load(file)
fdata1 = f'../data/post_data/perfect_toehold8/{nnn}/preprocess_perfect_toehold8.npz'
fdata2 = f'../data/post_data/perfect_toehold8/{nnn}/time_perfect_toehold8.npz'
print(load_data.keys())
data1 = np.load(fdata1, allow_pickle=True)
data2 = np.load(fdata2, allow_pickle=True)
print(data1.files, data2.files, load_data.keys())
trj_id = data2['trj_id']
indices_all = data1['indices_all']
hold_time_uniq = data2['hold_time_uniq']
trajs_states = load_data['trajs_states']
data1['dp_uniq'].shape, len(data1['indices_all']), data1['trans_time'].shape, data2['hold_time'].shape

dict_keys(['trajs_states', 'trajs_times', 'trajs_energies', 'ref_name'])
['dp_uniq', 'dp_og_uniq', 'energy_uniq', 'indices_uniq', 'indices_all', 'trans_time', 'ref_name'] ['hold_time', 'hold_time_uniq', 'cum_time_uniq', 'freq_uniq', 'trj_id'] dict_keys(['trajs_states', 'trajs_times', 'trajs_energies', 'ref_name'])


((13323,), 631176, (631176,), (631176,))

In [196]:
all_edges[0][0], all_edges[0][1]

(2381, 2264)

### Load hdf5 data

In [255]:
import h5py as h5
import numpy as np

def read_trajectory(traj_filename, sim_no): 
    with h5.File(traj_filename, "r") as f:
        times = f[str(sim_no)]["times"][:]
        energies = f[str(sim_no)]["energies"][:]
        structs = [s.decode() for s in f[str(sim_no)]["structs"]]
        ids = [s.decode() for s in f[str(sim_no)]["ordered_ids"]]
    return times, energies, structs, ids

def parse_cids(cids):
  # decode strand IDs
  sids = [np.array(c.split(b'+'), dtype=int) // 2 for c in cids]
  smin = np.concatenate(sids).min()
  sids = [(c - smin).tolist() for c in sids]
  # find minimal cyclic permutation per complex
  prm_parts = lambda c: lambda p: (c[p:len(c)], c[0:p])
  prm = lambda c: lambda p: sum(prm_parts(c)(p), start=[])
  argprm = lambda c: min(np.atleast_1d(np.argmin(c)), key=prm(c))
  return [tuple(prm(c)(argprm(c))) for c in sids]

# [parse_cids(cids.encode(encoding='utf-8').split(b' ')) for cids in [ids[0], ids[-1]]]

In [263]:
file = f'../data/raw_data/machinektest/perfect_toehold7_dangle.hdf5'
times, energies, structs, ids = read_trajectory(file, 5)

In [264]:
structs,[parse_cids(cids.encode(encoding='utf-8').split(b' ')) for cids in [ids[0], ids[-1]]]

(['................(((((((((((((((((+......(.........)(......+.........))))))))))))))))))',
  '................(((((((((((((((((+..........))))))))))))))))) ......(.........).......'],
 [[(0, 2, 1)], [(0, 1), (2,)]])

In [262]:
structs,[parse_cids(cids.encode(encoding='utf-8').split(b' ')) for cids in [ids[0], ids[-1]]]


(['.............(..(((((((((((((((((+..........)))))))))))))))))+.................)......',
  '................(((((((((((((((((+..........))))))))))))))))) ........................'],
 [[(0, 1, 2)], [(0, 1), (2,)]])

In [112]:
order_start = parse_cids(ids[0].encode(encoding='utf-8').split(b' '))
order_end = parse_cids(ids[-1].encode(encoding='utf-8').split(b' '))
order_start, order_end

([(0, 2, 1)], [(0, 1), (2,)])

In [115]:
# ll = []

ll.append(order_start)
ll

[[(0, 1), (2,)], [(0, 2, 1)]]

In [120]:
ll[0]

[(0, 1), (2,)]

In [111]:
order_end

[(0, 1), (2,), [...]]

In [52]:
order_end == [(0,1), (2,)]

True

In [44]:
order_start == (0,1,2)

True

In [ ]:
# 0 refers to the incumbent
# 1 refers to the substrate   
# 2 refers to the invader

# case 1:
# (0,1,2)

# case 2:
# (0,2,1)

# case 3:
# (0,1), (2,)

# case 4:
# (0,), (1,2)

# (0,2,1) --> a+b+c

In [59]:
import os
import tqdm as tqdm

def read_machinek(inpath, rxn, num_traj):
    # TODO: wait for correct data format
    def _read_trajectory_h5(fpath, sim_no): 
        import h5py as h5
        with h5.File(fpath, "r") as f:
            times = f[str(sim_no)]["times"][:]
            energies = f[str(sim_no)]["energies"][:]
            structs = [s.decode() for s in f[str(sim_no)]["structs"]]          
            ids = [s.decode() for s in f[str(sim_no)]["ordered_ids"]]
        return times, energies, structs, ids
    
    fpath = os.path.join(inpath, f"{rxn}.hdf5")
    
    trajs_states, trajs_times, trajs_energies, trajs_ids  = [],[],[], []

    for i in tqdm.tqdm(range(num_traj)):
        traj = _read_trajectory_h5(fpath, i)
        trajs_times.append(traj[0])
        trajs_energies.append(traj[1])
        trajs_states.append(traj[2])
        trajs_ids.append(traj[3])
    
    trajs_times = np.array(trajs_times, dtype=object)
    trajs_energies = np.array(trajs_energies, dtype=object)
    trajs_states = np.array(trajs_states, dtype=object)
    trajs_ids = np.array(trajs_ids, dtype=object)
        
    return trajs_states, trajs_times, trajs_energies, trajs_ids

In [63]:
inpath = f'../data/raw_data/machinektest/'
num_traj = 5
rxn = 'proximal_toehold7_dangle'
trajs_states, trajs_times, trajs_energies, trajs_ids = read_machinek(inpath, rxn, num_traj)

100%|██████████| 5/5 [00:02<00:00,  2.14it/s]


In [64]:
trajs_ids

array([list(['4+0+2', '0+2 4']),
       list(['10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+8+6', '10+

In [65]:
ids = trajs_ids[1]

In [66]:
order_start = parse_cids(ids[0].encode(encoding='utf-8').split(b' '))
order_end = parse_cids(ids[-1].encode(encoding='utf-8').split(b' '))
order_start, order_end

([(0, 2, 1)], [(0, 1), (2,)])

In [172]:
import re

def assign_base_names(sequence):
    split_sequence = re.split(r'\s|\+', sequence)
    base_names = []

    for strand_index, strand in enumerate(split_sequence):
        strand_names = []
        
        for base_index, base_type in enumerate(strand):
            strand_names.append(f'{chr(ord("a") + strand_index)}{base_index + 1}')
            
        base_names.append(strand_names)
    
    return base_names

In [173]:
strand_sub = 'CCCTCCACATTCAACCTCAAACTCACC' 
strand_incb = 'TGGTGTTTGTGGGTGTGGTGAGTTTGAGGTTGA'  
strand_inv = 'GGTGAGTTTGAGGTTCAATGTGGA'  
ref_strands = strand_incb + '+' + strand_inv + '+' + strand_sub
ref_name_list = assign_base_names(ref_strands)
ref_name = [item for sublist in ref_name_list for item in sublist]

In [247]:
import numpy as np
import pickle
import gzip

rxn = 'perfect_toehold7_dangle'

fdata0 = f'../data/post_data/{rxn}/{rxn}.pkl.gz'
with gzip.open(fdata0, 'rb') as file:
    load_data = pickle.load(file)
print(load_data.keys())

fdata1 = f'../data/post_data/{rxn}/preprocess_{rxn}.npz'
data1 = np.load(fdata1, allow_pickle=True)
print(data1.files)

fdata2 = f'../data/post_data/{rxn}/adjmat_{rxn}.npz'
data2 = np.load(fdata2, allow_pickle=True)
print(data2.files)

dict_keys(['trajs_states', 'trajs_times', 'trajs_energies', 'trajs_ids', 'ref_name_list'])
['dp_uniq', 'dp_og_uniq', 'energy_uniq', 'id_uniq', 'indices_uniq', 'indices_all', 'trans_time', 'ref_name_list']
['adj_uniq']


In [248]:
indices_all = data1['indices_all']
trajs_states = load_data['trajs_states']
trajs_ids = load_data['trajs_ids']
id_uniq = data1['id_uniq']
dp_og_uniq = data1['dp_og_uniq']
dp_uniq = data1['dp_uniq']
adj_uniq = data2['adj_uniq']
ref_name_list = data1['ref_name_list']
energy_uniq = data1['energy_uniq']
data1['dp_uniq'].shape, len(data1['indices_all']), data1['trans_time'].shape, indices_all.shape, adj_uniq.shape

((1454,), 7057, (7057,), (7057,), (1454, 84, 84))

In [249]:
ref_name_list

array([list(['a1', 'a2', 'a3', 'a4', 'a5', 'a6', 'a7', 'a8', 'a9', 'a10', 'a11', 'a12', 'a13', 'a14', 'a15', 'a16', 'a17', 'a18', 'a19', 'a20', 'a21', 'a22', 'a23', 'a24', 'a25', 'a26', 'a27', 'a28', 'a29', 'a30', 'a31', 'a32', 'a33']),
       list(['b1', 'b2', 'b3', 'b4', 'b5', 'b6', 'b7', 'b8', 'b9', 'b10', 'b11', 'b12', 'b13', 'b14', 'b15', 'b16', 'b17', 'b18', 'b19', 'b20', 'b21', 'b22', 'b23', 'b24', 'b25', 'b26', 'b27']),
       list(['c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9', 'c10', 'c11', 'c12', 'c13', 'c14', 'c15', 'c16', 'c17', 'c18', 'c19', 'c20', 'c21', 'c22', 'c23', 'c24'])],
      dtype=object)

In [250]:
dp_og_uniq[:10], dp_og_uniq[-10:]

(array(['((....).........)((((((((((((((((+.........(..(..(........+.)..)..)..)))))))))))))))).',
        '((.............))(((((((((((((((.+.....((..((((...........+.))).)))...))))))))))))))).',
        '((.............))(((((((((((((((.+.....((...(((...........+.)))..))...))))))))))))))).',
        '(...(..........))((((((((((((((..+.....((....((...........+..))..))....)))))))))))))).',
        '(...)(.(....).).(((((((((((((((((+.(.(...(........).......+.)).......)))))))))))))))))',
        '(...)(....).....(((((((((((((((((+....(...................+.........))))))))))))))))))',
        '(...)(....).....((((((((((((((((.+.....((...(((...........+.)))..))...))))))))))))))))',
        '(...)(....).....((((((((((((((((.+............((.(........+....).))...))))))))))))))))',
        '(...)(....).....(((((((((((((((..+.....((..(.((.(......)..+..)).)))....)))))))))))))))',
        '(...)(....).....(((((((((((((((..+.....((..(.((...........+..)).)))....)))))))))))))))'],
       dtype='<U86'

In [251]:
id_uniq[:10], id_uniq[-10:]

(array([list([(0, 2, 1)]), list([(0, 2, 1)]), list([(0, 2, 1)]),
        list([(0, 2, 1)]), list([(0, 2, 1)]), list([(0, 2, 1)]),
        list([(0, 2, 1)]), list([(0, 2, 1)]), list([(0, 2, 1)]),
        list([(0, 2, 1)])], dtype=object),
 array([list([(0, 2, 1)]), list([(0, 2, 1)]), list([(0, 2, 1)]),
        list([(0, 2, 1)]), list([(0, 2, 1)]), list([(0, 1), (2,)]),
        list([(0, 2, 1)]), list([(0, 2, 1)]), list([(0, 2, 1)]),
        list([(0, 2, 1)])], dtype=object))

In [254]:
dp_og_uniq[8:10], id_uniq[8:10]

(array(['(...)(....).....(((((((((((((((..+.....((..(.((.(......)..+..)).)))....)))))))))))))))',
        '(...)(....).....(((((((((((((((..+.....((..(.((...........+..)).)))....)))))))))))))))'],
       dtype='<U86'),
 array([list([(0, 2, 1)]), list([(0, 2, 1)])], dtype=object))

In [255]:
adj8 = adj_uniq[8]
adj9 = adj_uniq[9]
np.sum(np.abs(adj8-adj9))

2

In [256]:
dp_og_uniq[44], dp_og_uniq[45], id_uniq[44], id_uniq[45], dp_uniq[44], dp_uniq[45]

('(.......)(......(((((((((((((((((+..........)))))))))))))))))+....)..((.(...).))......',
 '(.......)(......(((((((((((((((((+..........)))))))))))))))))+..........).............',
 [(0, 1, 2)],
 [(0, 1, 2)],
 '(.......)(......(((((((((((((((((..........)))))))))))))))))....)..((.(...).))......',
 '(.......)(......(((((((((((((((((..........)))))))))))))))))..........).............')

In [257]:
adj44 = adj_uniq[44]
adj45 = adj_uniq[45]
np.sum(np.abs(adj44-adj45))

10

In [258]:
dp_og_uniq[43], dp_og_uniq[45], id_uniq[43], id_uniq[45], dp_uniq[43], dp_uniq[45]

('(.......)(....).(((((((((((((((((+.....((.................+......))..)))))))))))))))))',
 '(.......)(......(((((((((((((((((+..........)))))))))))))))))+..........).............',
 [(0, 2, 1)],
 [(0, 1, 2)],
 '(.......)(....).(((((((((((((((((.....((.......................))..)))))))))))))))))',
 '(.......)(......(((((((((((((((((..........)))))))))))))))))..........).............')

In [259]:
adj43 = adj_uniq[43]
adj45 = adj_uniq[45]
np.sum(np.abs(adj43-adj45))

8

In [262]:
np.sum(np.abs(adj43-adj44))

14

In [261]:
energy43 = energy_uniq[43]
energy44 = energy_uniq[44]
energy45 = energy_uniq[45]
energy43, energy44, energy45

(3.2782900371479613, 6.306242640726875, 3.056873165858935)

In [236]:
def solve_order(order_id, ref_name_list):
    """ 
    0 refers to the incumbent (33), denoted as "a"
    1 refers to the substrate (27), denoted as "b"
    2 refers to the invader (24),   denoted as "c"
    default ref_name_list order: a, b, c 
    
    """
    
    if order_id == [(0, 1, 2)] or order_id == [(0, 1),(2,)] or order_id == [(0,),(1, 2)]:
        alter_name = np.concatenate([ref_name_list[0], ref_name_list[1], ref_name_list[2]])
        print("case 1")
    
    elif order_id == [(0, 2, 1)] or order_id == [(0, 2),(1,)] or order_id == [(0,),(2, 1)]:
        alter_name = np.concatenate([ref_name_list[0], ref_name_list[2], ref_name_list[1]])
        print("case 2")
                
    else:
        print(order_id)
        raise ValueError("Invalid reaction ordering")
    
    return alter_name



# convert dot-parenthesis notation to adjacency matrix for three-strand
def dp2adj_3strand(ref_name, alter_name, dp_structure):
    # construct backbone edges
    def build_consecutive_edges(input_list):
        edges = [(input_list[i], input_list[i+1]) for i in range(len(input_list)-1)]
        
        return edges

    # build backbone edges
    all_backbone_edges = build_consecutive_edges(alter_name)
    # remove cross-strand edges
    backbones = []
    for edge in all_backbone_edges:
        # Only keep edges that connect nucleotides with the same prefix (a-a, b-b, c-c)
        if edge[0][0] == edge[1][0]:
            backbones.append(edge)

    # build base pair edges
    stack = []  # Initialize stack to keep track of opening brackets
    base_pairs = []  # Initialize list to store pairs    
    
    for name, char in zip(alter_name, dp_structure):
        
        if char == '(':
            stack.append(name)  # Push index of opening bracket onto stack
        elif char == ')':
            if stack:
                opening_index = stack.pop()  # Pop top index from stack
                base_pairs.append((opening_index, name))  # Create a pair
            else:
                print("Error: Mismatched brackets")
                return None
    
    if stack:
        print("Error: Mismatched brackets")
        return None
    
    # collect all edges
    all_pairs = backbones + base_pairs
 
    # assign nodes and edges
    nodes = ref_name.tolist() 
    edges = all_pairs 

    # Initialize adjacency matrix with zeros
    adjacency_matrix = np.zeros((len(nodes), len(nodes)), dtype=int)

    # Populate the adjacency matrix based on edges
    for edge in edges:
        i = nodes.index(edge[0])
        j = nodes.index(edge[1])
        adjacency_matrix[i, j] = 1
        adjacency_matrix[j, i] = 1

    return adjacency_matrix


In [237]:
ref_name_list

array([list(['a1', 'a2', 'a3', 'a4', 'a5', 'a6', 'a7', 'a8', 'a9', 'a10', 'a11', 'a12', 'a13', 'a14', 'a15', 'a16', 'a17', 'a18', 'a19', 'a20', 'a21', 'a22', 'a23', 'a24', 'a25', 'a26', 'a27', 'a28', 'a29', 'a30', 'a31', 'a32', 'a33']),
       list(['b1', 'b2', 'b3', 'b4', 'b5', 'b6', 'b7', 'b8', 'b9', 'b10', 'b11', 'b12', 'b13', 'b14', 'b15', 'b16', 'b17', 'b18', 'b19', 'b20', 'b21', 'b22', 'b23', 'b24', 'b25', 'b26', 'b27']),
       list(['c1', 'c2', 'c3', 'c4', 'c5', 'c6', 'c7', 'c8', 'c9', 'c10', 'c11', 'c12', 'c13', 'c14', 'c15', 'c16', 'c17', 'c18', 'c19', 'c20', 'c21', 'c22', 'c23', 'c24'])],
      dtype=object)

In [238]:
dp_og_uniq[43:45]

array(['(.......)(....).(((((((((((((((((+.....((.................+......))..)))))))))))))))))',
       '(.......)(......(((((((((((((((((+..........)))))))))))))))))+....)..((.(...).))......'],
      dtype='<U86')

In [239]:
dp_og_uniq[43], dp_og_uniq[44], dp_og_uniq[45]

('(.......)(....).(((((((((((((((((+.....((.................+......))..)))))))))))))))))',
 '(.......)(......(((((((((((((((((+..........)))))))))))))))))+....)..((.(...).))......',
 '(.......)(......(((((((((((((((((+..........)))))))))))))))))+..........).............')

In [241]:
order_id45 = id_uniq[45]
dp45 = dp_uniq[45]
alter_name45 = solve_order(order_id45, ref_name_list)
ref_name = np.concatenate(ref_name_list)
# print(ref_name)
print(dp_og_uniq[45])
print(dp45)
print(order_id45)
print(alter_name45)
aa = dp2adj_3strand(ref_name, alter_name45, dp45)
aa.shape

case 1
(.......)(......(((((((((((((((((+..........)))))))))))))))))+..........).............
(.......)(......(((((((((((((((((..........)))))))))))))))))..........).............
[(0, 1, 2)]
['a1' 'a2' 'a3' 'a4' 'a5' 'a6' 'a7' 'a8' 'a9' 'a10' 'a11' 'a12' 'a13'
 'a14' 'a15' 'a16' 'a17' 'a18' 'a19' 'a20' 'a21' 'a22' 'a23' 'a24' 'a25'
 'a26' 'a27' 'a28' 'a29' 'a30' 'a31' 'a32' 'a33' 'b1' 'b2' 'b3' 'b4' 'b5'
 'b6' 'b7' 'b8' 'b9' 'b10' 'b11' 'b12' 'b13' 'b14' 'b15' 'b16' 'b17' 'b18'
 'b19' 'b20' 'b21' 'b22' 'b23' 'b24' 'b25' 'b26' 'b27' 'c1' 'c2' 'c3' 'c4'
 'c5' 'c6' 'c7' 'c8' 'c9' 'c10' 'c11' 'c12' 'c13' 'c14' 'c15' 'c16' 'c17'
 'c18' 'c19' 'c20' 'c21' 'c22' 'c23' 'c24']


(84, 84)

In [242]:
order_id43 = id_uniq[43]
dp43 = dp_uniq[43]
alter_name43 = solve_order(order_id43, ref_name_list)
ref_name = np.concatenate(ref_name_list)
# print(ref_name)
print(dp_og_uniq[43])
print(dp43)
print(order_id43)
print(alter_name43)
bb = dp2adj_3strand(ref_name, alter_name43, dp43)
bb.shape

case 2
(.......)(....).(((((((((((((((((+.....((.................+......))..)))))))))))))))))
(.......)(....).(((((((((((((((((.....((.......................))..)))))))))))))))))
[(0, 2, 1)]
['a1' 'a2' 'a3' 'a4' 'a5' 'a6' 'a7' 'a8' 'a9' 'a10' 'a11' 'a12' 'a13'
 'a14' 'a15' 'a16' 'a17' 'a18' 'a19' 'a20' 'a21' 'a22' 'a23' 'a24' 'a25'
 'a26' 'a27' 'a28' 'a29' 'a30' 'a31' 'a32' 'a33' 'c1' 'c2' 'c3' 'c4' 'c5'
 'c6' 'c7' 'c8' 'c9' 'c10' 'c11' 'c12' 'c13' 'c14' 'c15' 'c16' 'c17' 'c18'
 'c19' 'c20' 'c21' 'c22' 'c23' 'c24' 'b1' 'b2' 'b3' 'b4' 'b5' 'b6' 'b7'
 'b8' 'b9' 'b10' 'b11' 'b12' 'b13' 'b14' 'b15' 'b16' 'b17' 'b18' 'b19'
 'b20' 'b21' 'b22' 'b23' 'b24' 'b25' 'b26' 'b27']


(84, 84)

In [243]:
np.sum(np.abs(aa - bb))

8

In [244]:
dp_og_uniq[43], dp_og_uniq[45], id_uniq[43], id_uniq[45]

('(.......)(....).(((((((((((((((((+.....((.................+......))..)))))))))))))))))',
 '(.......)(......(((((((((((((((((+..........)))))))))))))))))+..........).............',
 [(0, 2, 1)],
 [(0, 1, 2)])

In [245]:
dd45 = dp_og_uniq[45].split('+')[0] + "+" + dp_og_uniq[45].split('+')[2] + "+" + dp_og_uniq[45].split('+')[1]
dd45

'(.......)(......(((((((((((((((((+..........).............+..........)))))))))))))))))'

In [246]:
dp_og_uniq[43], dd45

('(.......)(....).(((((((((((((((((+.....((.................+......))..)))))))))))))))))',
 '(.......)(......(((((((((((((((((+..........).............+..........)))))))))))))))))')